# Data Processing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import os

os.makedirs('figures', exist_ok=True)

df = pd.read_csv('credit_approval/credit_approval.csv')

In [ ]:
# Preprocessing
from sklearn.preprocessing import LabelEncoder

df_clean = df.dropna().reset_index(drop=True)
print(f"Rows after dropping missing: {len(df_clean)}  (dropped {len(df) - len(df_clean)})")

CAT_COLS  = ['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12', 'A13']
CONT_COLS = ['A2', 'A3', 'A8', 'A11', 'A14', 'A15']
FEATURES  = CAT_COLS + CONT_COLS
TARGET    = 'A16'

df_enc = df_clean.copy()
encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col])
    encoders[col] = le

df_enc[TARGET] = (df_enc[TARGET] == '+').astype(int)

X             = df_enc[FEATURES].values.astype(float)
y             = df_enc[TARGET].values
feature_names = FEATURES

print(f"X shape: {X.shape}, class balance: {y.mean():.1%} approved (+)")

**Note:**
1. We drop all rows that contain N.A., because
   1. it only constitutes a very small fraction of the dataset
   2. imputation is risky when we do not know what each feature means
2. LabelEncoder() might be problematic for multi-category features because it artificially introduces ordering. Fortunately, none of these features contribute significantly (see later).

# Model Training

In [ ]:
# Model Selection via K-Fold CV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
import warnings; warnings.filterwarnings('ignore')

param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5],
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=1),
    param_distributions=param_dist,
    n_iter=60,
    scoring='roc_auc',
    cv=cv,
    return_train_score=False,
    n_jobs=-1,
    random_state=42,
    refit=False,
)
search.fit(X, y)

results    = pd.DataFrame(search.cv_results_)
AUC_FLOOR  = 0.80
candidates = results[results['mean_test_score'] > AUC_FLOOR]
best_idx   = candidates['std_test_score'].idxmin()
best_params = results.loc[best_idx, 'params']

print(f"Candidates above AUC >= {AUC_FLOOR}: {len(candidates)}")
print(f"Selected mean AUC: {results.loc[best_idx, 'mean_test_score']:.4f}  ",
      f"std: {results.loc[best_idx, 'std_test_score']:.4f}")
print(f"Params: {best_params}")

best_rf = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
best_rf.fit(X, y)

**Note:**
1. We want a model that is (1) fairly accurate and (2) stable.
2. The goal is not to maximize OOS prediction accuracy, but to identify which features drive decisions, in what direction, and how they interact.
3. We use a RandomForest model and randomized search to tune hyperparameters that yield the most stable results across folds while maintaining a high AUC.

# Feature Analysis and Interpretability

## SHAP and Feature Importance

In [ ]:
# Per-Fold: Permutation Importance + SHAP Signs
from sklearn.inspection import permutation_importance
import shap

fold_perm_imp   = []
fold_shap_signs = []

for k, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    rf = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)

    perm = permutation_importance(
        rf, X_val, y_val, scoring='roc_auc', n_repeats=5, random_state=42, n_jobs=-1
    )
    fold_perm_imp.append(perm.importances_mean)

    expl = shap.TreeExplainer(rf)
    sv   = expl.shap_values(X_val)
    sv1  = sv[1] if isinstance(sv, list) else sv[..., 1]
    fold_shap_signs.append(sv1.mean(axis=0))

    print(f"  fold {k+1:2d}/10", end='\r')

fold_perm_imp   = np.array(fold_perm_imp)
fold_shap_signs = np.array(fold_shap_signs)
print('Per-fold analysis complete.')

In [ ]:
# SHAP on Final Model (all data)
explainer_final = shap.TreeExplainer(best_rf)
sv_final        = explainer_final.shap_values(X)
shap_approval   = sv_final[1] if isinstance(sv_final, list) else sv_final[..., 1]

mean_abs_shap = np.abs(shap_approval).mean(axis=0)
top_idx       = np.argsort(mean_abs_shap)[::-1]
top_features  = [feature_names[i] for i in top_idx]
TOP_N = 8

print('Feature ranking by mean |SHAP|:')
for rank, i in enumerate(top_idx):
    print(f"  {rank+1:2d}. {feature_names[i]:4s}  {mean_abs_shap[i]:.4f}")

## Figure 1 -- SHAP Beeswarm Plot

In [ ]:
# Saves: figures/fig1_beeswarm.png
shap_exp = shap.Explanation(
    values=shap_approval,
    base_values=(
        explainer_final.expected_value[1]
        if isinstance(explainer_final.expected_value, (list, np.ndarray))
        else explainer_final.expected_value
    ),
    data=X,
    feature_names=feature_names,
)

plt.figure()
shap.plots.beeswarm(shap_exp, max_display=15, show=False)
plt.tight_layout()
plt.savefig('figures/fig1_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/fig1_beeswarm.png')

## Figure 2 -- Permutation Importance (Mean +/- Std)

In [ ]:
# Saves: figures/fig2_permutation.png
mean_imp = fold_perm_imp.mean(axis=0)
std_imp  = fold_perm_imp.std(axis=0)
order    = np.argsort(mean_imp)[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(feature_names))
ax.bar(x, mean_imp[order], yerr=std_imp[order], capsize=4,
       color='steelblue', alpha=0.8, error_kw=dict(ecolor='black', lw=1.2))
ax.set_xticks(x)
ax.set_xticklabels([feature_names[i] for i in order], rotation=45, ha='right')
ax.set_ylabel('Permutation importance (AUC drop)')
ax.set_title('Feature Importance -- Mean +/- Std Across 10 Folds')
plt.tight_layout()
plt.savefig('figures/fig2_permutation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/fig2_permutation.png')

## Figure 3 -- Pearson Correlation Heatmap (Top 8 Features)

In [ ]:
# Saves: figures/fig3_correlation.png
top8_features = [feature_names[i] for i in top_idx[:8]]
df_top8 = pd.DataFrame(X, columns=feature_names)[top8_features]
corr = df_top8.corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr, annot=True, fmt='.2f', center=0,
    cmap='RdBu_r', vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    annot_kws={'size': 9}
)
ax.set_title('Pearson Correlation -- Top 8 Features by Mean |SHAP|')
plt.tight_layout()
plt.savefig('figures/fig3_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/fig3_correlation.png')

## Figure 4 -- PDP + ICE Plots (A11, A8, A15, A14)

In [ ]:
# Saves: figures/fig4_pdp_ice.png
from sklearn.inspection import PartialDependenceDisplay
import random

TARGET_FEATURES = ['A11', 'A8', 'A15', 'A14']
feat_indices = [feature_names.index(f) for f in TARGET_FEATURES]

rng = random.Random(42)
ice_sample = rng.sample(range(len(X)), min(200, len(X)))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes_flat = axes.flatten()

for ax, feat_name, feat_idx in zip(axes_flat, TARGET_FEATURES, feat_indices):
    disp = PartialDependenceDisplay.from_estimator(
        best_rf, X, features=[feat_idx],
        kind='both',
        subsample=ice_sample,
        ax=ax,
        random_state=42,
        ice_lines_kw={'color': 'steelblue', 'alpha': 0.08, 'lw': 0.8},
        pd_line_kw={'color': 'black', 'lw': 2.5, 'ls': '--', 'label': 'PDP'},
    )
    ax.set_title(f'PDP + ICE: {feat_name}')
    ax.set_xlabel(feat_name)
    ax.set_ylabel('Predicted approval probability')

plt.suptitle('Partial Dependence + ICE Plots', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('figures/fig4_pdp_ice.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/fig4_pdp_ice.png')

## Figure 5 -- SHAP Sign Stability Matrix (Fold x Feature)

In [ ]:
# Saves: figures/fig5_sign_matrix.png
sign_matrix  = np.sign(fold_shap_signs)
sign_ordered = sign_matrix[:, top_idx]
top_labels   = [feature_names[i] for i in top_idx]

cmap = mcolors.LinearSegmentedColormap.from_list('rg', ['#e74c3c', '#f0f0f0', '#2ecc71'])

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(sign_ordered, aspect='auto', cmap=cmap, vmin=-1, vmax=1)

ax.set_xticks(range(len(top_labels)))
ax.set_xticklabels(top_labels, rotation=45, ha='right')
ax.set_yticks(range(10))
ax.set_yticklabels([f'Fold {k+1}' for k in range(10)])
ax.set_title('SHAP Direction Stability -- Sign of Mean SHAP per Fold')

for i in range(10):
    for j in range(len(top_labels)):
        val = sign_ordered[i, j]
        txt = '+' if val > 0 else ('-' if val < 0 else '0')
        ax.text(j, i, txt, ha='center', va='center', fontsize=10, fontweight='bold',
                color='white' if abs(val) == 1 else 'black')

plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
plt.tight_layout()
plt.savefig('figures/fig5_sign_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/fig5_sign_matrix.png')

## Verify All Figures Exist

In [ ]:
expected = [
    'figures/fig1_beeswarm.png',
    'figures/fig2_permutation.png',
    'figures/fig3_correlation.png',
    'figures/fig4_pdp_ice.png',
    'figures/fig5_sign_matrix.png',
]
for path in expected:
    status = 'OK      ' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}]  {path}')